# Early-exit pipeline, live check

`changed_src/judge_early_exit_pipeline.py` runs the standard queries one at a time and answers early when the T7 judge gives P >= 0.98 to the choice.
Check: run it live (GPU) on official photos from the dump set and compare, photo by photo, with the replay of the same rule on the stored dump
(`Ours_Judge_Check`, section 6). Same stage, same place, same number of queries expected (small differences possible: the dump stores matches at 1/64 px).

In [1]:
import os, sys, time
os.environ.setdefault("OMP_NUM_THREADS", "6")
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); sys.path.insert(0, str(ROOT / "tests/Test_Iteration_7/changed_src")); os.chdir(ROOT)
import json, joblib, numpy as np, pandas as pd
from train.checks import replay
from train.checks.candidate_judge import add_context, add_features, decide_frozen, judge_new
T7 = ROOT / "tests/Test_Iteration_7/results"
N_EARLY, N_OTHER = int(os.environ.get("N_EARLY", "8")), int(os.environ.get("N_OTHER", "5"))
table = add_context(add_features(pd.read_csv(T7 / "signals_ours_official150.csv.gz")))
frozen = json.load(open(T7 / "frozen_rules_t7.json")); p = judge_new(joblib.load(T7 / "judge_t7.joblib"), table)
expected = decide_frozen(table, p, frozen["variants"]["pc_early"]["tau"], "pc_early").set_index("photo")
ours = {pid: replay.decide_current(ph) for pid, ph in replay.load_set(T7 / "rich_dump/official150").items()}
early_ids = expected.index[expected.stage == "early"][:N_EARLY].tolist()
other_ids = [i for i in expected.index[expected.stage != "early"] if ours[i]["stage"] in ("escalation", "agreement", "skip")][:N_OTHER] + \
            [i for i in expected.index[expected.stage != "early"] if ours[i]["stage"] == "bar"][:2]
photo_ids = early_ids + other_ids
print(len(early_ids), "photos expected to exit early;", len(other_ids), "others (agreement / escalation / skip / bar)")

8 photos expected to exit early; 7 others (agreement / escalation / skip / bar)


In [2]:
from judge_early_exit_pipeline import JudgeEarlyExitPipeline
pipeline = JudgeEarlyExitPipeline("official", "edge_zncc", semantic_layer=True, semantic_decision=True, gamma=1.0, k_peaks=50, m_fine=10,
                                  matcher="loftr_minima", min_inliers=8, ransac_pixels=None, matcher_confidence=0.22, satellite="new",
                                  map_margin_m=None, loftr_coarse_threshold=0.21, dino_weight=None)
pipeline.setup(500, 7)
for line in pipeline.describe():
    print(line)
test_set = pipeline.test_set

level 3: 0.85 m/pixel, picture 16378 x 16005, map coverage 100%
level 4: 1.20 m/pixel, picture 11581 x 11317, map coverage 100%
level 5: 1.70 m/pixel, picture 8189 x 8002, map coverage 100%
level 6: 2.40 m/pixel, picture 5790 x 5658, map coverage 100%
level 7: 3.39 m/pixel, picture 4094 x 4001, map coverage 100%


map store ready, took 118 s


/home/voidness/Storage/Synced Folders/InterIIT Tech 15/venv-geoloc/lib/python3.10/site-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


/home/voidness/Storage/Synced Folders/InterIIT Tech 15/venv-geoloc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


data: official | 379 photos available | zoom levels [3, 4, 5, 6, 7]
photos: official | map: new
coarse method: edge_zncc | maps: ['edge_zncc']
semantic layer: on (best K=50 -> M=10) | DINO in the decision: on (gamma 1.00, rescue (None, None))
fine matcher: loftr_minima | answer bar: 8 inliers | RANSAC 3.0 px | matcher confidence 0.22 | LoFTR coarse threshold 0.21, upscale 1.0 | border 0 px, margin 0 px
DINO order weight: 100.0 | projection overrides: none (settings.GEOMETRY)
batches: 1 LoFTR pairs and 25 DINO places per call
device: cuda
queries per photo: 6 (headings [-5.0, 0.0, 5.0], scale factors [1.0, 1.05]); K and M are per query; 1 queries matched together
decision: answer at >= 8 inliers or >= 2 queries agreeing within 1.0 sigma (each >= 3 inliers)
escalation: on; 27 wide queries, answer at >= 11 inliers
early exit: answer after any of the first 5 standard queries when the T7 judge gives P >= 0.98 to the choice


In [3]:
rows = []
for pid in photo_ids:
    t0 = time.time()
    r = pipeline.run(test_set.items[pid], test_set.priors[pid])
    err = float(np.hypot(r["x"] - test_set.truths[pid][0], r["y"] - test_set.truths[pid][1]) / test_set.sigmas[pid])
    exp = expected.loc[pid]; o = ours[pid]
    rows.append(dict(photo=pid, live_stage=r["stage"], replay_stage=exp.stage if exp.stage == "early" else o["stage"], live_queries=r["n_queries_run"],
                     replay_queries=int(exp.n_queries) if exp.stage == "early" else o["n_queries"], live_err_sigma=round(err, 2),
                     replay_err_sigma=round(float(exp.err_sigma) if exp.stage == "early" else float(o["err_sigma"]), 2), p=round(r["judge_p"], 3),
                     seconds=round(time.time() - t0, 1)))
live = pd.DataFrame(rows)
live["same_stage"] = live.live_stage == live.replay_stage
live["same_place"] = (live.live_err_sigma - live.replay_err_sigma).abs() < 0.2
live

,photo,live_stage,replay_stage,live_queries,replay_queries,live_err_sigma,replay_err_sigma,p,seconds,same_stage,same_place
0,1,early,early,1,1,0.57,0.57,1.000,1.3,True,True
1,2,early,early,1,1,0.68,0.68,1.000,0.6,True,True
2,5,early,early,1,1,0.74,0.74,1.000,0.6,True,True
3,6,early,early,2,2,0.72,0.72,0.985,2.0,True,True
4,8,early,early,1,1,0.65,0.65,1.000,0.7,True,True
5,9,early,early,1,1,0.71,0.71,1.000,0.6,True,True
6,10,early,early,2,2,0.03,0.03,1.000,1.5,True,True
7,17,early,early,1,1,0.12,0.12,1.000,0.7,True,True
8,4,agreement,agreement,6,6,0.64,0.64,NaN,6.0,True,True
9,7,skip,skip,27,27,1.00,1.00,NaN,14.3,True,True


In [4]:
print("same stage:", int(live.same_stage.sum()), "of", len(live), "| same place:", int(live.same_place.sum()), "of", len(live))
print("mean live queries %.2f vs replay %.2f" % (live.live_queries.mean(), live.replay_queries.mean()))
live.to_csv(T7 / "early_exit_live_check.csv", index=False)

same stage: 15 of 15 | same place: 15 of 15
mean live queries 7.67 vs replay 7.67
